# Final Project - Intelligent Agents

## Phase 1 Code

In [ ]:
from simpleai.search import SearchProblem, astar, breadth_first, uniform_cost, iterative_limited_depth_first
import numpy as np
import time
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

mars_map = np.load("E:/Tec/cuarto/Agentes Inteligentes/Final Project/mars_map.npy")
mr, mc = mars_map.shape


class MarsRover(SearchProblem):
    # class of the problem
    def __init__(self, mars_map, start, goal):
        self.mars_map = mars_map
        self.goal = (*goal, mars_map[goal[0], goal[1]])
        initial_state = (*start, mars_map[start[0], start[1]])
        super().__init__(initial_state=initial_state)

    def actions(self, state):
        # when can we move
        r, c, z = state
        actions = []

        # Possible movements: Up, Down, Left, Right
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < mr and 0 <= nc < mc:
                nz = mars_map[nr, nc]
                if z != -1 and nz <= z + 2.5:
                    actions.append((nr, nc))

        return actions

    def result(self, state, action):
        # perform the action
        nr, nc = action
        return (nr, nc, mars_map[nr, nc])

    def is_goal(self, state):
        # check if we're in goal
        return state == self.goal

    def heuristic(self, state):
        # heuristic for A*, we use Manhattan distance
        Manhattan = abs(state[0] - self.goal[0]) + abs(state[1] - self.goal[1])
        return Manhattan


def get_path(result):
    """Extract list of (row, col) positions from a search result."""
    if result is None:
        return []
    path = []
    node = result
    while node:
        path.append((node.state[0], node.state[1]))
        node = node.parent
    return list(reversed(path))


def visualize_route(mars_map, start, goal, results_dict, title="Mars Rover Routes"):
    """
    Plot the mars map with routes overlaid.
    results_dict: {algorithm_name: result_node}
    """
    fig, ax = plt.subplots(figsize=(12, 8))

    ax.imshow(mars_map, cmap='terrain', origin='upper')

    colors = ['red', 'cyan', 'yellow', 'magenta']
    patches = []

    for (name, result), color in zip(results_dict.items(), colors):
        path = get_path(result)
        if path:
            rows, cols = zip(*path)
            ax.plot(cols, rows, color=color, linewidth=2, alpha=0.8)
            patches.append(mpatches.Patch(color=color, label=name))

    # Mark start and goal
    ax.plot(start[1], start[0], 'go', markersize=10, label='Start')
    ax.plot(goal[1], goal[0], 'r*', markersize=14, label='Goal')
    patches.append(mpatches.Patch(color='green', label='Start'))
    patches.append(mpatches.Patch(color='red', label='Goal'))

    ax.legend(handles=patches, loc='upper left')
    ax.set_title(title)
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")
    plt.tight_layout()
    plt.show()


def measure_algorithm(problem, algorithm_func, name):
    # to compare the algorithm's efficiency
    print(f"--- Measuring {name} ---")

    start_time = time.time()
    result = algorithm_func(problem, graph_search=True)
    end_time = time.time()

    execution_time = end_time - start_time

    if result:
        print("Status: Solution Found")

        # we get rid of the scale for this print to match the goal values
        print(f"Final State: x={result.state[1]*scale:.2f}, y={H*scale - result.state[0]*scale:.2f}, z={result.state[2]*scale:.2f}")
        print(f"Path Cost: {result.cost}")

    else:
        print("Status: No solution found")

    print(f"Execution Time: {execution_time:.5f} seconds\n")
    return result


scale = 10.0174
H = 1814

startx, starty = 2850, 6400
goalx, goaly = 3150, 6800

start_c = round(startx / scale)
start_r = round((H * scale - starty) / scale)

goal_c = round(goalx / scale)
goal_r = round((H * scale - goaly) / scale)

start = (start_r, start_c)
goal = (goal_r, goal_c)


problem = MarsRover(mars_map, start, goal)

print(f"Goal: {goal}")
print(f"Goal state: {problem.goal}")

print("")
print("Solve with both distances < 500")
print("")
r1 = {}
r1["BFS"] = measure_algorithm(problem, breadth_first, "Breadth-First Search")
r1["UCS"] = measure_algorithm(problem, uniform_cost, "Uniform-Cost Search")
r1["A*"] = measure_algorithm(problem, astar, "A* Search")
r1["IDS"] = measure_algorithm(problem, iterative_limited_depth_first, "Iterative Deepening Search")
visualize_route(mars_map, start, goal, r1, title="Routes: distances < 500")

scale = 10.0174
H = 1814

startx, starty = 2850, 5400
goalx, goaly = 3850, 6800

start_c = round(startx / scale)
start_r = round((H * scale - starty) / scale)

goal_c = round(goalx / scale)
goal_r = round((H * scale - goaly) / scale)

start = (start_r, start_c)
goal = (goal_r, goal_c)


problem = MarsRover(mars_map, start, goal)

print(f"Goal: {goal}")
print(f"Goal state: {problem.goal}")

print("")
print("Solve with distances from 1,000 to 5,000")
print("")
r2 = {}
r2["BFS"] = measure_algorithm(problem, breadth_first, "Breadth-First Search")
r2["UCS"] = measure_algorithm(problem, uniform_cost, "Uniform-Cost Search")
r2["A*"] = measure_algorithm(problem, astar, "A* Search")
#measure_algorithm(problem, iterative_limited_depth_first, "Iterative Deepening Search")
print("IDS took too long in this case and it couldn't find a path\n")
visualize_route(mars_map, start, goal, r2, title="Routes: distances 1,000–5,000")

scale = 10.0174
H = 1814

startx, starty = 3200, 1000
goalx, goaly = 3500, 11000

start_c = round(startx / scale)
start_r = round((H * scale - starty) / scale)

goal_c = round(goalx / scale)
goal_r = round((H * scale - goaly) / scale)

start = (start_r, start_c)
goal = (goal_r, goal_c)

problem = MarsRover(mars_map, start, goal)

print(f"Goal: {goal}")
print(f"Goal state: {problem.goal}")

print("")
print("Solve with distance x > 10,000")
print("")
r3 = {}
r3["BFS"] = measure_algorithm(problem, breadth_first, "Breadth-First Search")
#measure_algorithm(problem, uniform_cost, "Uniform-Cost Search")
print("UCS took too long in this case, it could find a path but took 76 seconds")
r3["A*"] = measure_algorithm(problem, astar, "A* Search")
#measure_algorithm(problem, iterative_limited_depth_first, "Iterative Deepening Search")
print("IDS took too long in this case and it couldn't find a path\n")
visualize_route(mars_map, start, goal, r3, title="Routes: distance x > 10,000")

## Phase 2 Code

In [ ]:
#------------------------------------------------------------------------------------------------------------------
#   Height map pre-processing
#------------------------------------------------------------------------------------------------------------------

#------------------------------------------------------------------------------------------------------------------
#   Imports
#------------------------------------------------------------------------------------------------------------------
import copy
import numpy as np
from skimage.transform import downscale_local_mean

import matplotlib.cm as cm
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource

import plotly.graph_objects as px

#------------------------------------------------------------------------------------------------------------------
#   File names
#------------------------------------------------------------------------------------------------------------------
input_file = "E:/Descargas/mars_map.IMG"
output_file = "mars_map.npy"

#------------------------------------------------------------------------------------------------------------------
#   Load map data
#------------------------------------------------------------------------------------------------------------------
n_rows = 10.045
n_columns = 10.045
data_file = open(input_file, "rb")

endHeader = False;
while not endHeader:
    line = data_file.readline().rstrip().lower()

    sep_line = line.split(b'=')

    if len(sep_line) == 2:
        itemName = sep_line[0].rstrip().lstrip()
        itemValue = sep_line[1].rstrip().lstrip()

        if itemName == b'valid_maximum':
            maxV = float(itemValue)
        elif itemName == b'valid_minimum':
            minV = float(itemValue)
        elif itemName == b'lines':
            n_rows = int(itemValue)
        elif itemName == b'line_samples':
            n_columns = int(itemValue)
        elif itemName == b'map_scale':
            scale_str = itemValue.split()
            if len(scale_str) > 1:
                scale = float(scale_str[0])

    elif line == b'end':
        endHeader = True
        char = 0
        while char == 0 or char == 32:
            char = data_file.read(1)[0]
        pos = data_file.seek(-1, 1)

image_size = n_rows*n_columns
data = data_file.read(4*image_size)

image_data = np.frombuffer(data, dtype=np.dtype('f'))
image_data = image_data.reshape((n_rows, n_columns))
image_data = np.array(image_data)
image_data = image_data.astype('float64')

image_data = image_data - minV;
image_data[image_data < -10000] = -1;

#------------------------------------------------------------------------------------------------------------------
#   Subsampling
#------------------------------------------------------------------------------------------------------------------
sub_rate = round(10/scale)

image_data = downscale_local_mean(image_data, (sub_rate, sub_rate))
image_data[image_data<0] = -1

print('Sub-sampling:', sub_rate)

new_scale = scale*sub_rate
print('New scale:', new_scale, 'meters/pixel')

#------------------------------------------------------------------------------------------------------------------
#   Save map
#------------------------------------------------------------------------------------------------------------------
np.save(output_file, image_data)

#------------------------------------------------------------------------------------------------------------------
#   Show 3D surface
#------------------------------------------------------------------------------------------------------------------

x = new_scale*np.arange(image_data.shape[1])
y = new_scale*np.arange(image_data.shape[0])
X, Y = np.meshgrid(x, y)

fig = px.Figure(data = px.Surface(x=X, y=Y, z=np.flipud(image_data), colorscale='hot', cmin = 0,
                           lighting = dict(ambient = 0.0, diffuse = 0.8, fresnel = 0.02, roughness = 0.4, specular = 0.2),
                           lightposition=dict(x=0, y=n_rows/2, z=2*maxV)),

                layout = px.Layout(scene_aspectmode='manual',
                                   scene_aspectratio=dict(x=1, y=n_rows/n_columns, z=max((maxV-minV)/x.max(), 0.2)),
                                   scene_zaxis_range = [0,maxV-minV])
                )

fig.show()

#------------------------------------------------------------------------------------------------------------------
#   Show surface image
#------------------------------------------------------------------------------------------------------------------

cmap = copy.copy(plt.cm.get_cmap('autumn'))
cmap.set_under(color='black')

ls = LightSource(315, 45)
rgb = ls.shade(image_data, cmap=cmap, vmin = 0, vmax = image_data.max(), vert_exag=2, blend_mode='hsv')

fig, ax = plt.subplots()

im = ax.imshow(rgb, cmap=cmap, vmin = 0, vmax = image_data.max(),
                extent =[0, scale*n_columns, 0, scale*n_rows],
                interpolation ='nearest', origin ='upper')

cbar = fig.colorbar(im, ax=ax)
cbar.ax.set_ylabel('Altura (m)')

plt.title('Superficie de Marte')
plt.xlabel('x (m)')
plt.ylabel('y (m)')

plt.show()

#------------------------------------------------------------------------------------------------------------------
#   Load processed map
#------------------------------------------------------------------------------------------------------------------

import numpy as np

crater_map = np.load("mars_map.npy")

n_rows, n_cols = crater_map.shape

scale = 10.045  # meters per pixel

print("Map shape:", crater_map.shape)
print("Scale:", scale, "m/pixel")

#------------------------------------------------------------------------------------------------------------------
#   Find global minimum
#------------------------------------------------------------------------------------------------------------------

min_pos = np.unravel_index(np.argmin(crater_map), crater_map.shape)
min_height = crater_map[min_pos]

print("\nGlobal minimum depth:", min_height, "m at pixel", min_pos)
print("Global minimum location:",
      "x =", min_pos[1]*scale,
      "y =", min_pos[0]*scale)

#------------------------------------------------------------------------------------------------------------------
#   Convert meters to pixel coordinates
#------------------------------------------------------------------------------------------------------------------

def meters_to_pixels(x, y):
    col = int(x / scale)
    row = int(y / scale)
    return row, col


#------------------------------------------------------------------------------------------------------------------
#   Greedy Search
#------------------------------------------------------------------------------------------------------------------

def greedy_search(crater_map, start_row, start_col, max_height_diff=2.0):

    n_rows, n_cols = crater_map.shape

    current_row = start_row
    current_col = start_col

    path = [(current_row, current_col)]

    neighbors = [(-1,-1),(-1,0),(-1,1),
                 (0,-1),(0,1),
                 (1,-1),(1,0),(1,1)]

    while True:

        current_height = crater_map[current_row,current_col]

        best_height = current_height
        best_pos = None

        for dr,dc in neighbors:

            nr = current_row + dr
            nc = current_col + dc

            if 0 <= nr < n_rows and 0 <= nc < n_cols:

                neighbor_height = crater_map[nr,nc]

                if neighbor_height < 0:
                    continue

                if abs(current_height - neighbor_height) > max_height_diff:
                    continue

                if neighbor_height < best_height:
                    best_height = neighbor_height
                    best_pos = (nr,nc)

        if best_pos is None:
            break

        current_row,current_col = best_pos
        path.append(best_pos)

    return path


#------------------------------------------------------------------------------------------------------------------
#   Simulated Annealing
#------------------------------------------------------------------------------------------------------------------

def simulated_annealing(crater_map, start_row, start_col,
                        max_height_diff=2.0,
                        T_init=500.0,
                        T_min=0.0001,
                        alpha=0.9995,
                        max_iter=100000):

    n_rows, n_cols = crater_map.shape

    current_row = start_row
    current_col = start_col
    current_height = crater_map[current_row,current_col]

    best_row = current_row
    best_col = current_col
    best_height = current_height

    path = [(current_row,current_col)]

    neighbors = [(-1,-1),(-1,0),(-1,1),
                 (0,-1),(0,1),
                 (1,-1),(1,0),(1,1)]

    T = T_init
    iteration = 0

    while T > T_min and iteration < max_iter:

        valid_neighbors = []

        for dr,dc in neighbors:

            nr = current_row + dr
            nc = current_col + dc

            if 0 <= nr < n_rows and 0 <= nc < n_cols:

                nh = crater_map[nr,nc]

                if nh >= 0 and abs(current_height - nh) <= max_height_diff:
                    valid_neighbors.append((nr,nc,nh))

        if not valid_neighbors:
            break

        idx = np.random.randint(len(valid_neighbors))
        nr,nc,nh = valid_neighbors[idx]

        delta = nh - current_height

        if delta <= 0:

            current_row,current_col = nr,nc
            current_height = nh
            path.append((current_row,current_col))

        else:

            prob = np.exp(-delta/T)

            if np.random.random() < prob:

                current_row,current_col = nr,nc
                current_height = nh
                path.append((current_row,current_col))

        if current_height < best_height:
            best_height = current_height
            best_row = current_row
            best_col = current_col

        T *= alpha
        iteration += 1

    return path,(best_row,best_col,best_height),iteration


#------------------------------------------------------------------------------------------------------------------
#   Test positions
#------------------------------------------------------------------------------------------------------------------

test_positions = [
    (3350,5800),
    (3500,5500),
    (2700,4800),
    (3000,5000),
    (3100,5200),
    (2500,5150)
]

print("\n============================================================================")
print("GREEDY SEARCH RESULTS")
print("============================================================================")

for x,y in test_positions:

    row,col = meters_to_pixels(x,y)

    start_height = crater_map[row,col]

    path = greedy_search(crater_map,row,col)

    end_row,end_col = path[-1]
    end_height = crater_map[end_row,end_col]

    steps = len(path)

    descent = start_height - end_height

    dist = np.sqrt((end_row-min_pos[0])**2 + (end_col-min_pos[1])**2)*scale

    print("\nStart: x=",x,"y=",y,"height=",round(start_height,2))
    print("End pixel:",(end_row,end_col))
    print("End height:",round(end_height,2))
    print("Steps:",steps)
    print("Descent:",round(descent,2))
    print("Distance to bottom:",round(dist,2),"m")


print("\n============================================================================")
print("SIMULATED ANNEALING RESULTS")
print("============================================================================")

for x,y in test_positions:

    row,col = meters_to_pixels(x,y)

    start_height = crater_map[row,col]

    path,best,iterations = simulated_annealing(crater_map,row,col)

    best_row,best_col,best_height = best

    descent = start_height - best_height

    dist = np.sqrt((best_row-min_pos[0])**2 + (best_col-min_pos[1])**2)*scale

    print("\nStart: x=",x,"y=",y,"height=",round(start_height,2))
    print("Best position:",(best_row,best_col))
    print("Best height:",round(best_height,2))
    print("Iterations:",iterations)
    print("Path length:",len(path))
    print("Descent:",round(descent,2))
    print("Distance to bottom:",round(dist,2),"m")

In [ ]:
#------------------------------------------------------------------------------------------------------------------
#   3D Visualization: 1. Greedy Search
#------------------------------------------------------------------------------------------------------------------
import plotly.graph_objects as go

# Data preparation
zona_3d = np.where(crater_map < 0, np.nan, crater_map)
n_rows, n_cols = crater_map.shape

# X and Y Axes
x_axis = scale * np.arange(n_cols)
y_axis = scale * np.arange(n_rows)
X_mesh, Y_mesh = np.meshgrid(x_axis, y_axis)

# --- FIGURE 1: GREEDY SEARCH ---
fig_greedy = go.Figure()

# Terrain surface
fig_greedy.add_trace(go.Surface(
    x=X_mesh, y=Y_mesh, z=zona_3d,
    colorscale='hot',
    colorbar=dict(title='Height (m)'),
    name='Greedy Terrain'
))

# Loop to plot Greedy trajectories
colores_g = ['cyan', 'lime', 'magenta', 'white', 'yellow', 'orange']
for i, (x_m, y_m) in enumerate(test_positions):
    # Using your exact functions
    r, c = meters_to_pixels(x_m, y_m)
    path_g = greedy_search(crater_map, r, c)

    # Plotting using your indices (row=p[0], column=p[1])
    # x = column * scale, y = row * scale (with visual offset)
    path_gx = [p[1] * scale for p in path_g]
    path_gy = [p[0] * scale for p in path_g] # Your original row coordinates
    path_gz = [crater_map[p[0], p[1]] + 3 for p in path_g] # Offset to avoid overlapping

    fig_greedy.add_trace(go.Scatter3d(
        x=path_gx, y=path_gy, z=path_gz,
        mode='lines',
        line=dict(color=colores_g[i % len(colores_g)], width=5),
        name=f'Greedy Start ({x_m}, {y_m})'
    ))

# Visualization settings (No Y-axis inversion to match your indices)
fig_greedy.update_layout(
    title='Results: Greedy Search',
    scene=dict(
        xaxis_title='Columns (m)',
        yaxis_title='Rows (m)',
        zaxis_title='Z (m)',
        aspectratio=dict(x=1, y=n_rows/n_cols, z=0.5)
    )
)

fig_greedy.show()


#------------------------------------------------------------------------------------------------------------------
#   3D Visualization: 2. Simulated Annealing
#------------------------------------------------------------------------------------------------------------------

# --- FIGURE 2: SIMULATED ANNEALING ---
fig_sa = go.Figure()

# Terrain surface
fig_sa.add_trace(go.Surface(
    x=X_mesh, y=Y_mesh, z=zona_3d,
    colorscale='hot',
    colorbar=dict(title='Height (m)'),
    name='SA Terrain'
))

# Loop to plot Simulated Annealing trajectories
colores_sa = ['lime', 'magenta', 'cyan', 'white', 'yellow', 'orange']
for i, (x_m, y_m) in enumerate(test_positions):
    r, c = meters_to_pixels(x_m, y_m)

    path_sa, best, iters = simulated_annealing(crater_map, r, c)

    # Plotting
    path_sx = [p[1] * scale for p in path_sa]
    path_sy = [p[0] * scale for p in path_sa]
    path_sz = [crater_map[p[0], p[1]] + 5 for p in path_sa]

    fig_sa.add_trace(go.Scatter3d(
        x=path_sx, y=path_sy, z=path_sz,
        mode='lines',
        line=dict(color=colores_sa[i % len(colores_sa)], width=4, dash='solid'),
        name=f'SA Start ({x_m}, {y_m})'
    ))

# Visualization settings
fig_sa.update_layout(
    title='Results: Simulated Annealing',
    scene=dict(
        xaxis_title='Columns (m)',
        yaxis_title='Rows (m)',
        zaxis_title='Z (m)',
        aspectratio=dict(x=1, y=n_rows/n_cols, z=0.5)
    )
)

fig_sa.show()